In [1]:
# import libraries

import pandas as pd
import os
from keplergl import KeplerGl
from pyproj import CRS
import numpy as np
from matplotlib import pyplot as plt

In [2]:
#set path
path = r'C:\Users\nrsmi\Documents\CareerFoundry\New_York_CitiBike_Analysis\Data\Prepared_Data'

In [3]:
# import df
df = pd.read_csv(os.path.join(path, 'reduced_data_to_plot_7.csv'), index_col = 0 )

In [4]:
df.head()

,start_station_name,end_station_name,start_lat,start_lng,end_lat,end_lng,avgTemp,bike_rides_daily,ride_duration
date,,,,,,,,,
2022-01-01,Cleveland Pl & Spring St,Mott St & Prince St,40.722104,-73.997249,40.723180,-73.994800,11.6,20428,25.322883
2022-01-01,Broadway & E 14 St,E 25 St & 1 Ave,40.734546,-73.990741,40.738177,-73.977387,11.6,20428,10.860767
2022-01-01,28 Ave & 35 St,Vernon Blvd & 30 Rd,40.767100,-73.916986,40.770845,-73.934171,11.6,20428,9.277000
2022-01-01,Bergen St & Smith St,Atlantic Ave & Furman St,40.686744,-73.990632,40.691652,-73.999979,11.6,20428,3.747900
2022-01-01,Schermerhorn St & Bond St,Smith St & 3 St,40.688417,-73.984517,40.678724,-73.995991,11.6,20428,5.517417


In [5]:
# Create a value column and group by start and end station 

df['value'] = 1
df_group = df.groupby(['start_station_name', 'end_station_name'])['value'].count().reset_index()

In [6]:
df_group.head()

,start_station_name,end_station_name,value
0,1 Ave & E 110 St,1 Ave & E 110 St,72
1,1 Ave & E 110 St,1 Ave & E 39 St,1
2,1 Ave & E 110 St,1 Ave & E 62 St,3
3,1 Ave & E 110 St,1 Ave & E 68 St,2
4,1 Ave & E 110 St,1 Ave & E 78 St,3


In [7]:
df_group.shape

(430962, 3)

In [8]:
# Create a new DataFrame with the necessary columns
df_stations = df[['start_station_name', 'end_station_name', 'start_lat', 'start_lng', 'end_lat', 'end_lng']]

In [9]:
# Merge the station df with the grouped df
df_stations = pd.merge(
    df_stations,
    df_group.rename(columns={'value': 'trips'}),
    on=['start_station_name', 'end_station_name'],
    how='left'  # Use 'left' to ensure all rows from df_stations are retained
)

In [10]:
df_stations.head()

,start_station_name,end_station_name,start_lat,start_lng,end_lat,end_lng,trips
0,Cleveland Pl & Spring St,Mott St & Prince St,40.722104,-73.997249,40.723180,-73.994800,28
1,Broadway & E 14 St,E 25 St & 1 Ave,40.734546,-73.990741,40.738177,-73.977387,31
2,28 Ave & 35 St,Vernon Blvd & 30 Rd,40.767100,-73.916986,40.770845,-73.934171,1
3,Bergen St & Smith St,Atlantic Ave & Furman St,40.686744,-73.990632,40.691652,-73.999979,48
4,Schermerhorn St & Bond St,Smith St & 3 St,40.688417,-73.984517,40.678724,-73.995991,11


In [11]:
df_stations.shape

(2379383, 7)

In [12]:
df_stations_dupes = df_stations[df_stations.duplicated(subset=['start_station_name', 'end_station_name', 'trips'])]

In [13]:
df_stations_dupes

,start_station_name,end_station_name,start_lat,start_lng,end_lat,end_lng,trips
36,West End Ave & W 94 St,Riverside Dr & W 91 St,40.794165,-73.974124,40.793135,-73.977004,34
49,Broadway & E 14 St,E 25 St & 1 Ave,40.734546,-73.990741,40.738177,-73.977387,31
88,1 Ave & E 39 St,E 39 St & 2 Ave,40.747140,-73.971130,40.748033,-73.973828,68
91,2 Ave & E 31 St,Cleveland Pl & Spring St,40.742909,-73.977061,40.722104,-73.997249,19
105,Union St & 4 Ave,4 Ave & 9 St,40.677274,-73.982820,40.670513,-73.988766,21
...,...,...,...,...,...,...,...
2379378,Cathedral Pkwy & Broadway,W 20 St & 8 Ave,40.804283,-73.967036,40.743453,-74.000040,2
2379379,Park Cir & East Dr,6 Ave & 9 St,40.651531,-73.972160,40.668127,-73.983776,16
2379380,Division St & Bowery,Sands St & Jay St,40.714193,-73.996732,40.700119,-73.986200,14
2379381,MacDonough St & Marcy Ave,Centre St & Chambers St,40.680808,-73.946169,40.712733,-74.004607,2


In [14]:
df_stations_cleaned = df_stations.drop_duplicates(subset=['start_station_name', 'end_station_name', 'trips'])

In [15]:
df_stations_cleaned.shape

(430962, 7)

In [16]:
#df_stations_cleaned.to_csv(os.path.join(path, 'reduced_data_StationsTrips.csv'))

In [17]:
df_stations_cleaned = df_stations_cleaned.sort_values(by='trips', ascending=False)
df_stations_cleaned.head(100)

,start_station_name,end_station_name,start_lat,start_lng,end_lat,end_lng,trips
4625,Central Park S & 6 Ave,Central Park S & 6 Ave,40.765909,-73.976342,40.765909,-73.976342,1001
3590,7 Ave & Central Park South,7 Ave & Central Park South,40.766741,-73.979069,40.766741,-73.979069,673
4289,Roosevelt Island Tramway,Roosevelt Island Tramway,40.757284,-73.953600,40.757284,-73.953600,654
7081,Grand Army Plaza & Central Park S,Grand Army Plaza & Central Park S,40.764397,-73.973715,40.764397,-73.973715,614
19310,Soissons Landing,Soissons Landing,40.692317,-74.014866,40.692317,-74.014866,577
...,...,...,...,...,...,...,...
226,S 3 St & Bedford Ave,N 6 St & Bedford Ave,40.712605,-73.962644,40.717452,-73.958509,223
35811,12 Ave & W 40 St,Pier 40 - Hudson River Park,40.760875,-74.002777,40.727714,-74.011296,220
220,45 Ave & 21 St,46 Ave & 5 St,40.747371,-73.947773,40.747310,-73.954510,220
4136,10 Ave & W 14 St,10 Ave & W 14 St,40.741982,-74.008316,40.741982,-74.008316,218


In [19]:
df_stations_cleaned = df_stations_cleaned.sort_values(by='trips', ascending=False).head(100000)

In [20]:
df_stations_cleaned.to_csv(os.path.join(path, 'reduced_data_StationsTrips.csv'))

In [21]:
df_stations_cleaned.shape

(100000, 7)

In [21]:
# Create KeplerGl instance

m = KeplerGl(height = 700, data={"data_1": df_stations_cleaned})
m

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


KeplerGl(data={'data_1':                        start_station_name                   end_station_name  \
4625 …

In [22]:
config = m.config

In [23]:
config

{'version': 'v1',
 'config': {'visState': {'filters': [{'dataId': ['data_1'],
     'id': '7ndrkomlg',
     'name': ['trips'],
     'type': 'range',
     'value': [60.06, 99],
     'plotType': 'histogram',
     'animationWindow': 'free',
     'yAxis': None,
     'view': 'side',
     'speed': 1,
     'enabled': True}],
   'layers': [{'id': 'txpux5',
     'type': 'point',
     'config': {'dataId': 'data_1',
      'label': 'start',
      'color': [69, 138, 70],
      'highlightColor': [252, 242, 26, 255],
      'columns': {'lat': 'start_lat', 'lng': 'start_lng'},
      'isVisible': True,
      'visConfig': {'radius': 10,
       'fixedRadius': False,
       'opacity': 0.8,
       'outline': False,
       'thickness': 2,
       'strokeColor': None,
       'colorRange': {'name': 'Global Warming',
        'type': 'sequential',
        'category': 'Uber',
        'colors': ['#5A1846',
         '#900C3F',
         '#C70039',
         '#E3611C',
         '#F1920E',
         '#FFC300']},
       's

In [24]:
import json
with open("config.json", "w") as outfile:
    json.dump(config, outfile)

In [25]:
m.save_to_html(file_name = 'Bike Trips Aggregated.html', read_only = False, config = config)

Map saved to Bike Trips Aggregated.html!
